In [1]:
%load_ext autoreload
%autoreload 2

import time
import pandas as pd

from eventflow.config import EXPERIMENT_LOG, TUNING_FOLD
from eventflow.data import load_train, make_train_examples
from eventflow.lgbm import LightGBMModel
from eventflow.validation import add_folds, cross_validate

train = load_train()
examples = add_folds(make_train_examples(train))
print(f"{len(examples):,} practice questions from {examples['sequence_id'].nunique():,} machines")

596,953 practice questions from 6,500 machines


In [2]:
# Tune on fold 0 only. Each run learns from ALL training questions of folds 1-4 (90% of those
# machines; the other 10% are held back for early stopping), not the 150,000 sample of the quick check.
# The first run also makes the features (about a minute); the later runs reuse them.
candidates = [
    {"num_leaves": 63},                              # the quick-check setting
    {"num_leaves": 31},                              # smaller trees
    {"num_leaves": 127},                             # bigger trees
    {"num_leaves": 63, "min_child_samples": 200},    # every leaf needs at least 200 examples (smoother) Every leaf must cover at least 200 questions, so the rules are based on more evidence. That gives smoother predictions and less memorising.
]
rows = []
for settings in candidates:
    start = time.time()
    result = cross_validate(lambda: LightGBMModel(**settings), train, examples, folds=[TUNING_FOLD])
    rows.append({**settings, "logloss": result.mean, "mimic": result.mimic,
                 "minutes": round((time.time() - start) / 60, 1)})
tuning = pd.DataFrame(rows)
best = candidates[tuning["logloss"].idxmin()]
print("Best settings:", best)
tuning.round(4)

  stopped at round 198
fold 0: logloss 1.4820  mimic 1.4810  (152.6 s)
  stopped at round 256
fold 0: logloss 1.4809  mimic 1.4812  (130.4 s)
  stopped at round 142
fold 0: logloss 1.4860  mimic 1.4803  (163.1 s)
  stopped at round 184
fold 0: logloss 1.4788  mimic 1.4795  (155.9 s)
Best settings: {'num_leaves': 63, 'min_child_samples': 200}


,num_leaves,logloss,mimic,minutes,min_child_samples
0,63,1.4820,1.4810,2.5,NaN
1,31,1.4809,1.4812,2.2,NaN
2,127,1.4860,1.4803,2.7,NaN
3,63,1.4788,1.4795,2.6,200.0


In [3]:
# The best settings on all 5 folds. Out-of-fold predictions are saved to outputs/oof/lgbm_all.parquet
# for the final comparison and the error analysis.
result = cross_validate(lambda: LightGBMModel(**best), train, examples, name="lgbm_all")
result

  stopped at round 184
fold 0: logloss 1.4788  mimic 1.4795  (159.1 s)
  stopped at round 172
fold 1: logloss 1.4799  mimic 1.4260  (144.8 s)
  stopped at round 172
fold 2: logloss 1.4832  mimic 1.5233  (142.2 s)
  stopped at round 195
fold 3: logloss 1.4842  mimic 1.4593  (164.5 s)
  stopped at round 201
fold 4: logloss 1.4785  mimic 1.4843  (164.6 s)


CVResult(lgbm_all: folds 0-4, logloss 1.4809 ± 0.0026, mimic 1.4745)

In [4]:
# Log to experiments/results.csv. Run this cell ONCE (re-running adds a duplicate row).
result.log(run_id="lgbm_all", author="Nam", model="LightGBM", feature_set="all 6 groups (110 features)",
           params={**LightGBMModel(**best).params, "patience": 50},
           notes="5-fold cross_validate; early stopping on 10% held-back training machines")
print(pd.read_csv(EXPERIMENT_LOG).tail(1).T)

                                                              6
timestamp                                   2026-11-28T17:15:13
run_id                                                 lgbm_all
author                                                      Nam
model                                                  LightGBM
feature_set                         all 6 groups (110 features)
folds                                                       0-4
val_logloss                                              1.4809
val_std                                                  0.0026
val_mimic                                               1.47448
val_folds          [1.47878, 1.47986, 1.4832, 1.48416, 1.47853]
test_logloss                                                NaN
params        {"learning_rate": 0.05, "min_child_samples": 2...
notes         5-fold cross_validate; early stopping on 10% h...
